#### Import dependencies

In [44]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams,Prefetch,PayloadFieldSchema,PointStruct,MatchAny,FieldCondition,Filter,PayloadSchemaType
import pandas as pd 
import numpy as np 
import json
import tiktoken

### Retrive all item from qdrant Collection

In [3]:
qdrant_client=QdrantClient(url="http://localhost:6333")

In [4]:
dummy_vector=np.zeros(1536).tolist()

In [5]:
payload=qdrant_client.query_points(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    query=dummy_vector,
    using="text-embedding-model-3-small",
    limit=1000,
    with_payload=['parent_asin'],
    with_vectors=False
)

In [6]:
payload

QueryResponse(points=[ScoredPoint(id=400, version=1, score=0.0, payload={'parent_asin': 'B09QGP16MY'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=497, version=1, score=0.0, payload={'parent_asin': 'B0B8P44G8M'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=297, version=1, score=0.0, payload={'parent_asin': 'B0BXRSFB93'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=364, version=1, score=0.0, payload={'parent_asin': 'B0BYKDMMVJ'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=124, version=1, score=0.0, payload={'parent_asin': 'B0BS3V1QBY'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=346, version=1, score=0.0, payload={'parent_asin': 'B0BL2HJ8WS'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=143, version=1, score=0.0, payload={'parent_asin': 'B0BMXG42VN'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=44, version=1, score=0.0, payload={'parent_asin': 'B0BLCMRKR

In [7]:
parent_asin_list=[items.payload["parent_asin"] for items in payload.points]

In [8]:
len(parent_asin_list)

1000

In [11]:

df_reviews = pd.read_json("C:\\Users\\jaysi\\Desktop\\Desktop\\Ai-engineering\\data\\meta_Electronics_2022_23_with_categeory_rating_100_sample_2000.jsonl", lines=True)


In [12]:
df_reviews.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[],169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...",[],24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN


In [13]:
len(df_reviews)

2000

In [14]:
df_reviews_sample=df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [15]:
df_reviews_sample

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN
5,All Electronics,"Double Din Car Stereo with Backup Camera, 7 In...",4.0,191,[【Phone Mirror Link】Car stereo support mirror ...,[],39.93,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'TRASH', 'url': 'https://www.amazon...",LSLYA,"[Electronics, Car & Vehicle Electronics, Car E...",{'Package Dimensions': '8.19 x 6.1 x 5.04 inch...,B0BGR8FS29,NaN,NaN,NaN
8,All Electronics,Aigo 32GB Digital Voice Recorder 3072kbps one-...,4.0,174,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'aigo 32GB Digital Voice Recorder 3...,aigo,"[Electronics, Portable Audio & Video, Digital ...","{'Item Weight': '46 Grams', 'Item model number...",B09YD1FZDN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1992,Camera & Photo,4K Smoke Detector WiFi Hidden Cameras HD 1080P...,2.7,238,[Full HD camera: 1080P HD resolution allows yo...,[],49.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Does it REALLY WORK? I put it to ...,Thanmiral,"[Electronics, Camera & Photo, Video Surveillan...",{'Package Dimensions': '5 x 4.92 x 1.54 inches...,B09KV7HQK9,NaN,NaN,NaN
1994,Cell Phones & Accessories,"for DJI OM 6 Magsafe Adapter Clamp, Compatible...",4.5,346,[Compatible Gimbal Stabilizer: for DJI Osmo Mo...,[],15.95,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'The Perfect Fix To Replace Stock C...,Aerbeis,"[Electronics, Camera & Photo, Accessories, Pro...",{'Package Dimensions': '2.87 x 2.8 x 0.39 inch...,B0BNF5NF31,NaN,NaN,NaN
1995,Office Products,"WALI Single Monitor Stand, Adjustable Gas Spri...",4.2,210,[Compatibility: The single monitor arm fits mo...,[],47.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'WALI Single Monitor Stand White (G...,WALI,"[Electronics, Computers & Accessories, Compute...","{'Manufacturer': 'WALI', 'Brand': 'WALI', 'Ite...",B09RTDQ8LZ,NaN,NaN,NaN
1997,All Electronics,BenQ TH585P 1080p Home Entertainment Projector...,4.5,327,[1080P RESOLUTION: 1080p Full HD image quality...,[Turn any space into your private arena. TH585...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'TH671ST - Low Input Lag', 'url': '...",BenQ,"[Electronics, Video Projectors]",{'Product Dimensions': '18.1 x 13.5 x 7 inches...,B09WF96S4C,NaN,NaN,NaN


In [16]:
len(df_reviews_sample)

1000

### define Function To Preprocess Review Data

In [23]:
def preprocess_review_data(row):
    return f"{row['title']} {row['description']}"
  

In [24]:
encoading=tiktoken.encoding_for_model("text-embedding-3-small")

In [19]:
encoading.encode("can i get some earPhone")

[4919, 602, 636, 1063, 2487, 7084]

In [29]:
def toek_count(row,model="text-embedding-3-small"):
    encoading=tiktoken.encoding_for_model(model)
    return len(encoading.encode(row["preprocessed_data"]))

In [26]:
df_reviews_sample['preprocessed_data']=df_reviews_sample.apply(preprocess_review_data,axis=1)

In [30]:
df_reviews_sample["preprocessd_data_token_count"]=df_reviews_sample.apply(toek_count,axis=1)

In [31]:
df_reviews_sample.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_data,preprocessd_data_token_count
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",77
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",34
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN,USB C Docking Station Dual Monitor for MacBook...,58
5,All Electronics,"Double Din Car Stereo with Backup Camera, 7 In...",4.0,191,[【Phone Mirror Link】Car stereo support mirror ...,[],39.93,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'TRASH', 'url': 'https://www.amazon...",LSLYA,"[Electronics, Car & Vehicle Electronics, Car E...",{'Package Dimensions': '8.19 x 6.1 x 5.04 inch...,B0BGR8FS29,NaN,NaN,NaN,"Double Din Car Stereo with Backup Camera, 7 In...",45
8,All Electronics,Aigo 32GB Digital Voice Recorder 3072kbps one-...,4.0,174,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'aigo 32GB Digital Voice Recorder 3...,aigo,"[Electronics, Portable Audio & Video, Digital ...","{'Item Weight': '46 Grams', 'Item model number...",B09YD1FZDN,NaN,NaN,NaN,Aigo 32GB Digital Voice Recorder 3072kbps one-...,31


In [32]:
len(df_reviews_sample)

1000

In [35]:
token_counts=df_reviews_sample["preprocessd_data_token_count"].sum()

In [36]:
token_counts

np.int64(93799)

### Create a new Qdrant Collection For reviews

In [43]:
qdrant_client.create_collection(
    collection_name="Amazon-item-collection-01-reviews",
    vectors_config=VectorParams(size=1536,distance=Distance.COSINE)
)

True

In [45]:
qdrant_client.create_payload_index(
    collection_name="Amazon-item-collection-01-reviews",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [46]:
def get_embedding(text,model="text-embedding-3-small"):
    response=openai.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

In [47]:
def get_embedding_batch(text_list,model="text-embedding-3-small",batch_size=100):
    if len(text_list)<=batch_size:
        response=openai.embeddings.create(input=text_list,model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings=[]
    counter=1
    for i in range(0,len(text_list),batch_size):
        batch=text_list[i:i+batch_size]
        response=openai.embeddings.create(input=batch,model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        counter+=1
    
    return all_embeddings

### Embedded The Text